# U2-02 · La neurona, sus activaciones y la retropropagación

*Programación de Redes Neuronales - Machine Learning (IFCD48) · Unidad 2, apartados 2.2 y 2.3*

Ejecuta cada celda con **Mayús + Intro**. Todo funciona en el navegador, sin instalar nada.

## 1. Funciones de activación y sus derivadas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import erf

z = np.linspace(-4, 4, 401)
funciones = {
    "sigmoide": 1 / (1 + np.exp(-z)),
    "tanh": np.tanh(z),
    "ReLU": np.maximum(z, 0),
    "GELU": z * 0.5 * (1 + erf(z / np.sqrt(2))),
}
fig, (a, b) = plt.subplots(1, 2, figsize=(4.4, 1.9))
for (nombre, f), ls in zip(funciones.items(), ["-", "-", "-", "--"]):
    a.plot(z, f, ls, label=nombre)
    b.plot(z, np.gradient(f, z), ls)         # derivada numérica
a.set_title("Función"); b.set_title("Derivada")
a.legend(fontsize=6)
plt.show()  # fig.savefig("activaciones.pdf")
der_sig = np.gradient(funciones["sigmoide"], z)
print(f"derivada máxima de la sigmoide: {der_sig.max():.3f}")
print(f"derivada de la sigmoide en z = 4: {der_sig[-1]:.4f}")
print(f"tras 10 capas: 0.25**10 = {0.25 ** 10:.1e}")

**Prueba.** Calcula cuánto queda de la señal tras 10 capas de tanh, cuya derivada máxima es 1, si cada capa trabaja en z = 1.

## 2. Capacidad: la espiral con 2, 8 y 64 neuronas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier

def espiral(n=300, ruido=0.25, semilla=0):
    """Dos brazos de espiral entrelazados: clases 0 y 1."""
    rng = np.random.default_rng(semilla)
    t = np.sqrt(rng.uniform(0, 1, n)) * 3 * np.pi
    clase = rng.integers(0, 2, n)
    r = t * np.where(clase == 1, 1, -1)
    X = np.column_stack([r * np.cos(t), r * np.sin(t)]) / 3
    return X + rng.normal(0, ruido, X.shape), clase

if __name__ == "__main__":       # solo al ejecutar este archivo
    X, y = espiral()
    Xp, yp = espiral(semilla=1)            # otra muestra para probar
    xx, yy = np.meshgrid(np.linspace(-3.5, 3.5, 200),
                         np.linspace(-3.5, 3.5, 200))
    fig, ejes = plt.subplots(1, 3, figsize=(4.4, 1.6))
    for eje, n in zip(ejes, [2, 8, 64]):
        red = MLPClassifier((n,), activation="tanh", solver="lbfgs",
                            max_iter=5000,
                            random_state=0).fit(X, y)
        zona = red.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1]
        eje.contourf(xx, yy, zona.reshape(xx.shape),
                     levels=[0, .5, 1], colors=["#CFE0F2", "#F6DDBB"])
        eje.scatter(*X.T, c=np.where(y, "#E08A1E", "#1F5FA8"), s=2)
        eje.set_title(f"{n} neuronas", loc="center")
        eje.set_xticks([]); eje.set_yticks([])
        print(f"{n:2d} neuronas ocultas: entrenamiento "
              f"{red.score(X, y):.3f}, "
              f"prueba {red.score(Xp, yp):.3f}")
    plt.show()  # fig.savefig("espiral.pdf")

En el navegador funciona otra versión de scikit-learn, y los resultados con 8 y 64 neuronas pueden diferir en unas centésimas de los del libro (en una prueba dieron 0.893/0.810 y 1.000/0.893). Las conclusiones no cambian.

**Prueba.** Cambia `ruido` a 0.1 y a 0.5 en las dos llamadas a `espiral`. ¿Cuántas neuronas hacen falta ahora?

## 3. Retropropagación a mano

Usa la función `espiral` de la celda anterior. El entrenamiento tarda unos segundos.

In [ ]:
import numpy as np

X, y = espiral()
y = y.reshape(-1, 1).astype(float)

def iniciar(semilla=0):
    rng = np.random.default_rng(semilla)
    return {"W1": rng.normal(0, 1, (2, 16)), "b1": np.zeros(16),
            "W2": rng.normal(0, 0.25, (16, 1)), "b2": np.zeros(1)}

def adelante(p, X):
    H = np.tanh(X @ p["W1"] + p["b1"])            # capa oculta
    P = 1 / (1 + np.exp(-(H @ p["W2"] + p["b2"])))  # salida
    return H, P

def perdida(p, X=X, y=y):
    _, P = adelante(p, X)
    return -np.mean(y * np.log(P) + (1 - y) * np.log(1 - P))

def atras(p, X=X, y=y):
    """Retropropagación: la regla de la cadena, de la salida
    hacia la entrada."""
    H, P = adelante(p, X)
    dZ2 = (P - y) / len(X)                 # error en la salida
    dH = dZ2 @ p["W2"].T                   # se reparte hacia atrás
    dZ1 = dH * (1 - H ** 2)                # derivada de tanh
    return {"W2": H.T @ dZ2, "b2": dZ2.sum(0),
            "W1": X.T @ dZ1, "b1": dZ1.sum(0)}

if __name__ == "__main__":
    p = iniciar()
    # comprobación: derivada numérica de tres pesos al azar
    g, h = atras(p), 1e-6
    for nombre, idx in [("W1", (0, 3)), ("W1", (1, 7)),
                        ("W2", (5, 0))]:
        p[nombre][idx] += h; f1 = perdida(p)
        p[nombre][idx] -= 2 * h; f2 = perdida(p)
        p[nombre][idx] += h
        print(f"{nombre}{idx}: retropropagación "
              f"{g[nombre][idx]:+.6f}, "
              f"numérica {(f1 - f2) / (2 * h):+.6f}")

    # entrenamiento: descenso de gradiente con estos gradientes
    for paso in range(1, 20_001):
        for k, v in atras(p).items():
            p[k] -= 0.5 * v
        if paso in (1, 100, 1000, 20_000):
            acierto = np.mean((adelante(p, X)[1] > 0.5) == y)
            print(f"paso {paso:5d}: pérdida {perdida(p):.4f}, "
                  f"acierto {acierto:.3f}")

**Ejercicio.** Introduce un error a propósito en `atras` (por ejemplo, quita `(1 - H ** 2)`) y vuelve a ejecutar la celda. ¿Lo detecta la comprobación del gradiente? ¿Y el entrenamiento?

> Rumelhart, D. E., Hinton, G. E. y Williams, R. J. (1986). Learning representations by back-propagating errors. *Nature*, 323, 533–536. https://doi.org/10.1038/323533a0
>
> Cybenko, G. (1989). Approximation by superpositions of a sigmoidal function. *Mathematics of Control, Signals and Systems*, 2(4), 303–314. https://doi.org/10.1007/BF02551274